# QuantJourney SDK - Authentication Methods

This notebook demonstrates different authentication methods for the QuantJourney API:

1. **Direct token/API key** - directly use a pre-obtained token
2. **Username/Password login** - authenticate with credentials
3. **Environment variables** - configuration via .env

## Prerequisites

Make sure you have:
- Access to QuantJourney API (https://api.quantjourney.cloud)
- Valid credentials (email/password or API key)

In [ ]:
# Minimum import - all you need to get started
import sys
sys.path.insert(0, '..')

import os
import json

from quantjourney.sdk import QuantJourneyAPI


In [ ]:
def format_error(e: Exception) -> str:
    """
    Format error for display, extracting JSON error body if available.
    Import APIError only when needed for error handling.
    """
    from quantjourney.sdk import APIError
    
    if isinstance(e, APIError):
        if hasattr(e, 'response_body') and e.response_body:
            return json.dumps(e.response_body, indent=2)
        error_info = {"error": str(e).split('[')[0].strip()}
        if getattr(e, 'request_id', None):
            error_info["request_id"] = e.request_id
        if getattr(e, 'error_code', None):
            error_info["code"] = e.error_code
        return json.dumps(error_info, indent=2)
    
    if hasattr(e, 'response') and e.response is not None:
        try:
            return json.dumps(e.response.json(), indent=2)
        except (ValueError, AttributeError):
            pass
    
    return str(e)

print("✓ Helper function format_error() defined")


In [ ]:
# Health Check - test API accessibility
qj = QuantJourneyAPI()

try:
    health = qj.health()
    print("API Health Check:")
    print(json.dumps(health, indent=2))
except Exception as e:
    print("API unavailable:")
    print(format_error(e))


## Method 1: API Key Authentication (Recommended)

API keys are permanent tokens prefixed with `qj_live_` or `qj_test_`.
The tenant is embedded in the key - no need to specify `tenant_id`.

In [ ]:
# METHOD 1: API Key Authentication (Recommended)

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

try:
    # Simply pass the API key - default URL is https://api.quantjourney.cloud
    qj = QuantJourneyAPI(api_key=API_KEY)
    
    # Test connection - verify authentication worked
    whoami = qj.auth.whoami()
    
    if not whoami.get('authenticated', False):
        raise Exception(whoami.get('message', 'Not authenticated'))

    print("SUCCESS - Authenticated with API key!")
    print(json.dumps(whoami, indent=2))

except Exception as e:
    print("FAILED:")
    print(format_error(e))


## Method 2: Login with Username/Password

Authenticate using email and password - the SDK automatically manages tokens.

In [ ]:
# METHOD 2: Username/Password Login
# Returns JWT tokens with 15 minute expiry - good for interactive sessions

credentials = {
    "email": "user@example.com",
    "password": "change_me",
    "tenant_id": "default"
}

print(f"Credentials: {credentials['email']} / {'*' * len(credentials['password'])} @ {credentials['tenant_id']}")

try:
    qj_login = QuantJourneyAPI()
    tokens = qj_login.auth.login(**credentials)

    print("\nSUCCESS - Login successful!")
    print(f"  Token expires in: {tokens.expires_in}s")
    print(f"  Access Token: {tokens.access_token[:50]}...")
    if tokens.refresh_token:
        print(f"  Refresh Token: {tokens.refresh_token[:50]}...")

except Exception as e:
    print("\nFAILED:")
    print(format_error(e))


In [ ]:
# Check logged-in user information
try:
    whoami = qj.auth.whoami()
    print("User information:")
    for k, v in whoami.items():
        print(f"  {k}: {v}")
except Exception as e:
    print(f"✗ Error: {e}")


In [ ]:
# Token refresh (automatic on 401, but can be done manually)
try:
    new_tokens = qj.auth.refresh()
    print(f"✓ Token refreshed!")
except Exception as e:
    print(f"✗ Refresh error: {e}")


## Method 3: Environment Variables

Configuration via environment variables - ideal for CI/CD and production.

In [ ]:
# METHOD 3: Environment Variables
# Supports both API key and username/password via environment:
# - QJ_API_KEY or QJ_ACCESS_TOKEN or QJ_TOKEN: API key
# - QJ_USER_ID + QJ_PASSWORD + QJ_TENANT_ID: Login credentials
# - QJ_API: Custom API URL (optional, defaults to https://api.quantjourney.cloud)

# Option A: API Key (preferred)
# os.environ['QJ_API_KEY'] = 'qj_...'

# Option B: Username/Password
os.environ['QJ_USER_ID'] = 'user@example.com'
os.environ['QJ_PASSWORD'] = 'change_me'
os.environ['QJ_TENANT_ID'] = 'default'

print(f"Environment: QJ_USER_ID={os.environ.get('QJ_USER_ID')}")

try:
    qj_env = QuantJourneyAPI.from_env()
    print("\nSUCCESS - Client created from environment variables!")
    
except Exception as e:
    print("\nFAILED:")
    print(format_error(e))


In [ ]:
# Example with .env file
# Create a .env file in your project directory:

dotenv_example = """
# .env file example
QJ_API=https://api.quantjourney.cloud
QJ_API_KEY=qj_your_api_key_here
QJ_TENANT_ID=default

# OR for login credentials:
# QJ_USER_ID=user@example.com
# QJ_PASSWORD=change_me
"""

print(".env file example:")
print(dotenv_example)


## Summary

**Import patterns:**
```python
# Minimum - all you need
from quantjourney.sdk import QuantJourneyAPI

# With error handling (optional)
from quantjourney.sdk import QuantJourneyAPI, APIError
```

**Response Format:**
- SUCCESS: Returns full JSON response from API
- FAILED: Returns structured JSON error with `type`, `title`, `status`, `detail`, `error_code`, `request_id`